# 04 - Text Feature Engineering

Extract interpretable, predictive features from the free-text pros and cons fields using topic modelling, complementing the structured sub-rating features prepared in data preparation stage.

## 4.1 Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time

# BERTopic and dependencies 
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# Project utilities
import sys
sys.path.insert(0, "../src")
from utils import (
    save_fig, set_plot_style,
    SEED, FIG_DIR,
    clean_text, get_company_patterns, remove_companies
)

set_plot_style()
np.random.seed(SEED)

print("Setup complete.")

c:\Users\anama\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Setup complete.


In [3]:
# Load splits from data preparation stage
train = pd.read_csv("../data/processed/train.csv")
val   = pd.read_csv("../data/processed/val.csv")
test  = pd.read_csv("../data/processed/test.csv")

print(f"Train: {train.shape}")
print(f"Val:   {val.shape}")
print(f"Test:  {test.shape}")

Train: (440112, 28)
Val:   (94310, 28)
Test:  (94310, 28)


## 4.2 Text Cleaning

Clean `pros` and `cons` into new columns `pros_clean` and `cons_clean`.  
Steps: replace newlines → lowercase → strip whitespace → collapse multiple spaces → fill NaN → remove company names.

Company names (extracted from the `firm` column + common abbreviations) are stripped to prevent BERTopic from discovering company-specific topics instead of substantive workplace themes.

In [4]:
# Build company name pattern from already-loaded data
firms_raw = train["firm"].dropna().unique()
company_pattern = get_company_patterns(firms_raw)
print(f"Company name patterns: {len(firms_raw)} firms + abbreviations")

# Apply text cleaning pipeline to all splits
for df, name in [(train, "Train"), (val, "Val"), (test, "Test")]:
    df["pros_clean"] = remove_companies(clean_text(df["pros"]), company_pattern)
    df["cons_clean"] = remove_companies(clean_text(df["cons"]), company_pattern)
    print(f"{name}: added pros_clean, cons_clean")

Company name patterns: 428 firms + abbreviations
Train: added pros_clean, cons_clean
Val: added pros_clean, cons_clean
Test: added pros_clean, cons_clean


In [5]:
# Before/after examples — show both basic cleaning and company removal
print("=== PROS: before vs after full cleaning pipeline ===\n")

# Find rows where company removal changed something (more interesting than just lowercase)
company_changed = train["pros"].fillna("").str.lower() != train["pros_clean"]
sample_idx = train[company_changed].sample(5, random_state=42).index

for i in sample_idx:
    original = train.loc[i, "pros"]
    cleaned = train.loc[i, "pros_clean"]
    print(f"Row {i}")
    print(f"  BEFORE: {str(original)[:120]!r}")
    print(f"  AFTER:  {cleaned[:120]!r}")
    print()

=== PROS: before vs after full cleaning pipeline ===

Row 358119
  BEFORE: 'Amazing team and great learning opportunities\r\nPeople always ready to help\r\nAbove 75% in terms of salaries compared to o'
  AFTER:  'amazing team and great learning opportunities people always ready to help above 75% in terms of salaries compared to oth'

Row 237313
  BEFORE: 'Flexible hours\r\nGood Pay\r\nGreat company culture'
  AFTER:  'flexible hours good pay great company culture'

Row 306919
  BEFORE: 'Benefits are great\r\nProduct is good\r\nA lot of the people that work there are great'
  AFTER:  'benefits are great product is good a lot of the people that work there are great'

Row 234084
  BEFORE: '- Fantastic perks\r\n- Great culture\r\n- Excellent level of repsonsibility and exposure'
  AFTER:  '- fantastic perks - great culture - excellent level of repsonsibility and exposure'

Row 24239
  BEFORE: 'On-site bonus, world-wide business trips, health care insurance, 3 extra vocation days, IT equipm

## 4.3 Sentence Transformer Embeddings

Encode `pros_clean` and `cons_clean` separately using `all-MiniLM-L6-v2` (384-dim each).  
Pros and cons saved as separate embedding arrays to preserve sentiment context.

In [6]:

# Load sentence transformer model (GPU-accelerated encoding)
model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda", local_files_only=True)
print(f"Model loaded: {model.get_sentence_embedding_dimension()}-dim, device: cuda")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: 384-dim, device: cuda


In [7]:
# Generate embeddings for pros and cons separately
pros_embeddings = {}
cons_embeddings = {}

for df, name in [(train, "train"), (val, "val"), (test, "test")]:
    print(f"\nEncoding {name} ({len(df)} rows)...")
    
    pros_emb = model.encode(
        df["pros_clean"].tolist(), show_progress_bar=True, device="cuda"
    )
    cons_emb = model.encode(
        df["cons_clean"].tolist(), show_progress_bar=True, device="cuda"
    )
    
    pros_embeddings[name] = pros_emb
    cons_embeddings[name] = cons_emb
    print(f"  {name}: pros {pros_emb.shape}, cons {cons_emb.shape}")


Encoding train (440112 rows)...


Batches:   0%|          | 0/13754 [00:00<?, ?it/s]

Batches:   0%|          | 0/13754 [00:00<?, ?it/s]

  train: pros (440112, 384), cons (440112, 384)

Encoding val (94310 rows)...


Batches:   0%|          | 0/2948 [00:00<?, ?it/s]

Batches:   0%|          | 0/2948 [00:00<?, ?it/s]

  val: pros (94310, 384), cons (94310, 384)

Encoding test (94310 rows)...


Batches:   0%|          | 0/2948 [00:00<?, ?it/s]

Batches:   0%|          | 0/2948 [00:00<?, ?it/s]

  test: pros (94310, 384), cons (94310, 384)


## 4.4 Save Embeddings

Save pros and cons embeddings separately (384-dim each) as `.npy` files.

In [9]:
PROCESSED_DIR = "../data/processed"

for name in ["train", "val", "test"]:
    pros_path = os.path.join(PROCESSED_DIR, f"{name}_pros_embeddings.npy")
    cons_path = os.path.join(PROCESSED_DIR, f"{name}_cons_embeddings.npy")
    np.save(pros_path, pros_embeddings[name])
    np.save(cons_path, cons_embeddings[name])
    print(f"Saved: {name}_pros_embeddings.npy {pros_embeddings[name].shape}")
    print(f"Saved: {name}_cons_embeddings.npy {cons_embeddings[name].shape}")

Saved: train_pros_embeddings.npy (440112, 384)
Saved: train_cons_embeddings.npy (440112, 384)
Saved: val_pros_embeddings.npy (94310, 384)
Saved: val_cons_embeddings.npy (94310, 384)
Saved: test_pros_embeddings.npy (94310, 384)
Saved: test_cons_embeddings.npy (94310, 384)


## 4.5 BERTopic Setup

Confirm embeddings and dataframes are ready for topic modelling.

In [10]:
# Verify embeddings are ready
print("Embedding shapes (expect 384 dimensions):")
for name in ["train", "val", "test"]:
    pros_shape = pros_embeddings[name].shape
    cons_shape = cons_embeddings[name].shape
    status = "✓" if pros_shape[1] == 384 and cons_shape[1] == 384 else "✗"
    print(f"  {status} {name}: pros {pros_shape}, cons {cons_shape}")

Embedding shapes (expect 384 dimensions):
  ✓ train: pros (440112, 384), cons (440112, 384)
  ✓ val: pros (94310, 384), cons (94310, 384)
  ✓ test: pros (94310, 384), cons (94310, 384)


In [12]:
# Confirm dataframes have pros_clean and cons_clean columns
for df, name in [(train, "Train"), (val, "Val"), (test, "Test")]:
    has_pros = "pros_clean" in df.columns
    has_cons = "cons_clean" in df.columns
    print(f"{name}: {len(df)} rows, pros_clean={has_pros}, cons_clean={has_cons}")

# Verify row counts match
for name, df in [("train", train), ("val", val), ("test", test)]:
    assert len(df) == pros_embeddings[name].shape[0], f"{name} row mismatch!"
    assert len(df) == cons_embeddings[name].shape[0], f"{name} row mismatch!"
print("✓ All row counts match")

Train: 440112 rows, pros_clean=True, cons_clean=True
Val: 94310 rows, pros_clean=True, cons_clean=True
Test: 94310 rows, pros_clean=True, cons_clean=True
✓ All row counts match


## 4.6 BERTopic Parameter Search

**Problem:** HDBSCAN can produce high outlier rates with default BERTopic settings.  
**Goal:** Find parameters that produce 10–50 meaningful topics with <50% outliers.

**Strategy:** Test 6 key configurations varying HDBSCAN cluster size and UMAP neighbourhood parameters on a 5K sample. Stop early once target is met.

Alternative approaches (different embedding models, combined pros+cons text) were considered as fallbacks but not required.

In [13]:
# Parameter search helper
def try_config(docs, embeddings, umap_params, hdbscan_params, label=""):
    """Fit BERTopic with given params. Returns (n_topics, outlier_pct, model)."""
    umap_model = UMAP(**umap_params, random_state=SEED)
    hdbscan_model = HDBSCAN(**hdbscan_params, prediction_data=True)
    
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        calculate_probabilities=False,  # Skip for speed during search
        verbose=False
    )
    
    t0 = time.time()
    topics, _ = topic_model.fit_transform(docs, embeddings=embeddings)
    elapsed = time.time() - t0
    
    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    outlier_pct = sum(1 for t in topics if t == -1) / len(topics) * 100
    
    print(f"[{label}] Topics: {n_topics}, Outliers: {outlier_pct:.1f}%, Time: {elapsed:.0f}s")
    print(f"  UMAP: n_neighbors={umap_params['n_neighbors']}, n_components={umap_params['n_components']}")
    print(f"  HDBSCAN: min_cluster_size={hdbscan_params['min_cluster_size']}, min_samples={hdbscan_params.get('min_samples', 'default')}")
    
    return n_topics, outlier_pct, topic_model

In [14]:
# Prepare 10K sample of 20+ word reviews for fast iteration
train["pros_words"] = train["pros_clean"].str.split().str.len()
long_mask = train["pros_words"] >= 20
long_idx = train[long_mask].index.values

np.random.seed(SEED)
sample_idx = np.random.choice(long_idx, size=min(10000, len(long_idx)), replace=False)
sample_docs = train.loc[sample_idx, "pros_clean"].tolist()
sample_emb = pros_embeddings["train"][sample_idx]

print(f"Sample: {len(sample_docs)} reviews (20+ words)")
print(f"Embeddings: {sample_emb.shape}")

Sample: 10000 reviews (20+ words)
Embeddings: (10000, 384)


In [16]:
# Define 6 key configurations
configs = [
    # Vary HDBSCAN (fixed UMAP defaults)
    {"label": "1-default", 
     "umap": {"n_neighbors": 15, "n_components": 5, "min_dist": 0.0, "metric": "cosine"},
     "hdbscan": {"min_cluster_size": 15}},
    
    {"label": "2-permissive", 
     "umap": {"n_neighbors": 15, "n_components": 5, "min_dist": 0.0, "metric": "cosine"},
     "hdbscan": {"min_cluster_size": 10, "min_samples": 5}},
    
    {"label": "3-larger", 
     "umap": {"n_neighbors": 15, "n_components": 5, "min_dist": 0.0, "metric": "cosine"},
     "hdbscan": {"min_cluster_size": 50, "min_samples": 10}},
    
    # Vary UMAP (with larger cluster HDBSCAN)
    {"label": "4-tight", 
     "umap": {"n_neighbors": 5, "n_components": 5, "min_dist": 0.0, "metric": "cosine"},
     "hdbscan": {"min_cluster_size": 50, "min_samples": 10}},
    
    {"label": "5-broad", 
     "umap": {"n_neighbors": 30, "n_components": 5, "min_dist": 0.0, "metric": "cosine"},
     "hdbscan": {"min_cluster_size": 50, "min_samples": 10}},
    
    {"label": "6-moredims", 
     "umap": {"n_neighbors": 15, "n_components": 10, "min_dist": 0.0, "metric": "cosine"},
     "hdbscan": {"min_cluster_size": 50, "min_samples": 10}},
]

# Run all configs 
print("=" * 60)
print("PARAMETER SEARCH")
print("=" * 60)

results = []

for cfg in configs:
    n_topics, outlier_pct, model = try_config(
        sample_docs, sample_emb, 
        cfg["umap"], cfg["hdbscan"], 
        label=cfg["label"]
    )
    results.append({
        "label": cfg["label"], 
        "n_topics": n_topics, 
        "outlier_pct": outlier_pct,
        "umap": cfg["umap"],
        "hdbscan": cfg["hdbscan"]
    })
    print()

# Pick best (lowest outlier rate that meets topic target)
valid = [r for r in results if r["n_topics"] >= 10 and r["outlier_pct"] < 50]
if valid:
    best_result = min(valid, key=lambda x: x["outlier_pct"])
else:
    best_result = min(results, key=lambda x: x["outlier_pct"])
    
best_config = {"umap": best_result["umap"], "hdbscan": best_result["hdbscan"]}

PARAMETER SEARCH
[1-default] Topics: 64, Outliers: 52.6%, Time: 21s
  UMAP: n_neighbors=15, n_components=5
  HDBSCAN: min_cluster_size=15, min_samples=default

[2-permissive] Topics: 124, Outliers: 48.5%, Time: 20s
  UMAP: n_neighbors=15, n_components=5
  HDBSCAN: min_cluster_size=10, min_samples=5

[3-larger] Topics: 30, Outliers: 52.1%, Time: 20s
  UMAP: n_neighbors=15, n_components=5
  HDBSCAN: min_cluster_size=50, min_samples=10

[4-tight] Topics: 42, Outliers: 35.2%, Time: 15s
  UMAP: n_neighbors=5, n_components=5
  HDBSCAN: min_cluster_size=50, min_samples=10

[5-broad] Topics: 3, Outliers: 0.3%, Time: 27s
  UMAP: n_neighbors=30, n_components=5
  HDBSCAN: min_cluster_size=50, min_samples=10

[6-moredims] Topics: 4, Outliers: 0.5%, Time: 24s
  UMAP: n_neighbors=15, n_components=10
  HDBSCAN: min_cluster_size=50, min_samples=10



**Outcome:** Config 4-tight (n_neighbors=5) achieved the best balance with 42 topics and 35.2% outliers. The `n_neighbors` parameter controls how UMAP defines local structure — smaller values preserve finer distinctions between reviews, producing more distinct topic clusters. Larger values (e.g., 30) caused topics to collapse into just 3 broad clusters. Final config: UMAP(n_neighbors=5, n_components=5) + HDBSCAN(min_cluster_size=50, min_samples=10).

## 4.7 Fit BERTopic on Pros

Fit on 20+ word pros reviews, then transform all training pros to get probability features.

The dataset is heavily skewed toward large companies. Without balancing, topics would disproportionately reflect dominant employers. We cap reviews per company to ensure diverse workplace themes are represented while retaining enough data (~10K+) for topic modeling.

**Config:** CountVectorizer (English stopwords) + UMAP(5, 5, 0.0, cosine) + HDBSCAN(50, 10) + calculate_probabilities=True

UMAP and HDBSCAN parameters selected from parameter search in section 4.6 (config 4-tight).

In [17]:
# Determine cap per company

# Add embedding index for tracking
train["emb_idx"] = range(len(train))

# Filter to 20+ word reviews (pros_words already computed in 4.6)
pros_20plus = train[train["pros_words"] >= 20].copy()

train["cons_words"] = train["cons_clean"].str.split().str.len()
cons_20plus = train[train["cons_words"] >= 20].copy()

# Test different caps
print("Reviews remaining at different caps:")
print(f"{'Cap':<6} {'Pros':>10} {'Cons':>10}")
print("-" * 50)
for cap in [20, 25, 30, 35, 40, 45, 50, 55, 60, 65]:
    pros_capped = pros_20plus.groupby('firm').apply(
        lambda x: x.sample(min(len(x), cap), random_state=42)
    ).reset_index(drop=True)
    
    cons_capped = cons_20plus.groupby('firm').apply(
        lambda x: x.sample(min(len(x), cap), random_state=42)
    ).reset_index(drop=True)
    
    print(f"Cap {cap:<4} {len(pros_capped):>10,} {len(cons_capped):>10,}")

# Distribution of reviews per company (before capping)
print("\nReviews per company (20+ words):")
print("=" * 50)
print(f"{'Metric':<10} {'Pros':>12} {'Cons':>12}")
print("-" * 50)
pros_dist = pros_20plus['firm'].value_counts()
cons_dist = cons_20plus['firm'].value_counts()
for stat in ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']:
    p_val = pros_dist.describe()[stat]
    c_val = cons_dist.describe()[stat]
    print(f"{stat:<10} {p_val:>12,.0f} {c_val:>12,.0f}")



Reviews remaining at different caps:
Cap          Pros       Cons
--------------------------------------------------
Cap 20        7,372      7,464
Cap 25        8,849      9,014
Cap 30       10,232     10,480
Cap 35       11,535     11,864
Cap 40       12,775     13,179
Cap 45       13,951     14,449
Cap 50       15,053     15,662
Cap 55       16,097     16,813
Cap 60       17,075     17,927
Cap 65       18,006     19,001

Reviews per company (20+ words):
Metric             Pros         Cons
--------------------------------------------------
count               417          422
mean                240          305
std                 622          789
min                   1            1
25%                  20           21
50%                  53           64
75%                 159          199
max               6,040        7,694


**Decision:** Cap at 55 for pros, 65 for cons.
- Pros median = 53 → cap at 55 (~1× median), yields ~16K reviews
- Cons median = 64 → cap at 65 (~1× median), yields ~19K reviews
- Caps set near respective medians to balance company diversity with data retention

In [ ]:
# Cap per company for balanced representation
max_per_company = 55  

pros_capped = pros_20plus.groupby('firm').apply(
    lambda x: x.sample(min(len(x), max_per_company), random_state=42)
).reset_index(drop=True)

# Extract docs and embeddings using tracked indices
docs_long_pros = pros_capped["pros_clean"].tolist()
emb_long_pros = pros_embeddings["train"][pros_capped["emb_idx"].values]

print(f"20+ word pros: {len(pros_20plus)} ({len(pros_20plus)/len(train)*100:.1f}%)")
print(f"After capping at {max_per_company}/company: {len(pros_capped)} ({len(pros_capped)/len(train)*100:.1f}%)")

20+ word pros: 99989 (22.7%)
After capping at 55/company: 16097 (3.7%)


In [ ]:
# Fit BERTopic on pros
vectorizer = CountVectorizer(stop_words='english')
umap_model = UMAP(n_neighbors=5, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, min_samples=10, prediction_data=True)

pros_topic_model = BERTopic(
    vectorizer_model=vectorizer,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True
)

print("Fitting BERTopic on 20+ word pros...")
t0 = time.time()
pros_topics_fit, pros_probs_fit = pros_topic_model.fit_transform(docs_long_pros, embeddings=emb_long_pros)
elapsed = time.time() - t0

n_topics = len(set(pros_topics_fit)) - (1 if -1 in pros_topics_fit else 0)
n_outliers = sum(1 for t in pros_topics_fit if t == -1)
print(f"\nFit complete in {elapsed:.0f}s")
print(f"Topics discovered: {n_topics}")
print(f"Outliers: {n_outliers} ({n_outliers/len(pros_topics_fit)*100:.1f}%)")

# Show all pros topics
print(f"\nAll {n_topics} Pros topics:")
for _, row in pros_topic_model.get_topic_info().iterrows():
    if row["Topic"] != -1:
        print(f"  Topic {row['Topic']:3d}: {row['Count']:5d} docs - {row['Name']}")

2026-02-28 16:32:44,073 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Fitting BERTopic on 20+ word pros...


2026-02-28 16:32:53,138 - BERTopic - Dimensionality - Completed ✓
2026-02-28 16:32:53,141 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-28 16:32:56,594 - BERTopic - Cluster - Completed ✓
2026-02-28 16:32:56,604 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-28 16:32:57,243 - BERTopic - Representation - Completed ✓



Fit complete in 14s
Topics discovered: 55
Outliers: 6453 (40.1%)

All 55 Pros topics:
  Topic   0:  1018 docs - 0_pay_benefits_good_salary
  Topic   1:   664 docs - 1_university_campus_research_students
  Topic   2:   546 docs - 2_discount_store_retail_staff
  Topic   3:   539 docs - 3_place_work_great_location
  Topic   4:   430 docs - 4_company_great_work_benefits
  Topic   5:   398 docs - 5_senior_management_leadership_team
  Topic   6:   323 docs - 6_pension_scheme_leave_good
  Topic   7:   308 docs - 7_training_apprenticeship_support_new
  Topic   8:   302 docs - 8_culture_company_people_great
  Topic   9:   285 docs - 9_london_uk_office_good
  Topic  10:   269 docs - 10_flexible_hours_working_work
  Topic  11:   265 docs - 11_team_teams_great_friendly
  Topic  12:   222 docs - 12_hotel_hotels_travelodge_hospitality
  Topic  13:   219 docs - 13_job_enjoyed_loved_worked
  Topic  14:   208 docs - 14_pros_pro_think_working
  Topic  15:   201 docs - 15_people_colleagues_friendly_work

In [20]:
# Inspect outlier documents to verify they're not systematically different
outlier_mask = np.array(pros_topics_fit) == -1
outlier_docs = pros_capped.loc[outlier_mask, "pros_clean"].tolist()
outlier_lengths = pros_capped.loc[outlier_mask, "pros_words"].values

print(f"Outliers: {len(outlier_docs)} ({len(outlier_docs)/len(pros_topics_fit)*100:.1f}%)")
print(f"\nOutlier word counts:")
print(f"  Min: {outlier_lengths.min()}, Max: {outlier_lengths.max()}, Median: {np.median(outlier_lengths):.0f}")

print(f"\nSample outlier reviews:")
print("=" * 60)
for doc in np.random.choice(outlier_docs, size=min(10, len(outlier_docs)), replace=False):
    print(f"  - {doc[:100]}...")
    print()

Outliers: 6453 (40.1%)

Outlier word counts:
  Min: 20, Max: 922, Median: 31

Sample outlier reviews:
  - there are lots of pro’s to working for . some of them include: a generous holiday allowance of 25 da...

  - pay was better than some call centers; all work time was not spent on the phone. processing applicat...

  - i have been working for scs now for 9 years and have been a sales manager for 6 years. scs is a grea...

  - being on a casual contract is absolute great when: - you are a student -have other commitments -woul...

  - wholeheartedly agree with the previous review from april. firstly, you absolutely cannot fault the g...

  - progressive, allows colleagues to develop their career through bespoke training tailored to the indi...

  - colleagues are very supportive and job security is good. able to work flexi time and it is good to w...

  - i've been with the company for almost 5 years and have been blessed with the opportunity to manage, ...

  - have some good staff a

In [21]:
# Verify outliers still receive meaningful topic probabilities
outlier_idx = np.where(np.array(pros_topics_fit) == -1)[0][0]
outlier_probs = pros_probs_fit[outlier_idx]

print(f"Outlier review: {docs_long_pros[outlier_idx][:100]}...")
print(f"\nTop 5 topic probabilities:")
topic_info = pros_topic_model.get_topic_info()
top_topics = np.argsort(outlier_probs)[::-1][:5]
for t in top_topics:
    name = topic_info.loc[topic_info['Topic'] == t, 'Name'].values[0]
    print(f"  Topic {t}: {outlier_probs[t]:.3f} - {name}")

Outlier review: afh feels like family. i left afh last year for what i thought was a better career opportunity, but ...

Top 5 topic probabilities:
  Topic 4: 0.043 - 4_company_great_work_benefits
  Topic 13: 0.030 - 13_job_enjoyed_loved_worked
  Topic 21: 0.025 - 21_years_worked_ive_company
  Topic 28: 0.019 - 28_values_customer_core_products
  Topic 44: 0.019 - 44_team_im_feel_support


In [22]:
# Transform ALL training pros to get probability features
print(f"Transforming all {len(train)} training pros...")
t0 = time.time()
all_pros_topics, all_pros_probs = pros_topic_model.transform(
    train["pros_clean"].tolist(),
    embeddings=pros_embeddings["train"]
)
elapsed = time.time() - t0

print(f"Transform complete in {elapsed:.0f}s")
print(f"\n--- PROS SUMMARY ---")
print(f"Number of pros topics: {len(pros_topic_model.get_topic_info()) - 1}")
print(f"Probability matrix shape: {np.array(all_pros_probs).shape}")
n_outliers = sum(1 for t in all_pros_topics if t == -1)
print(f"Outliers: {n_outliers} ({n_outliers/len(all_pros_topics)*100:.1f}%)")

2026-02-28 16:37:08,189 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.


Transforming all 440112 training pros...


2026-02-28 16:40:05,364 - BERTopic - Dimensionality - Completed ✓
2026-02-28 16:40:05,366 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-02-28 16:40:30,634 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-02-28 16:42:43,521 - BERTopic - Probabilities - Completed ✓
2026-02-28 16:42:43,522 - BERTopic - Cluster - Completed ✓


Transform complete in 336s

--- PROS SUMMARY ---
Number of pros topics: 55
Probability matrix shape: (440112, 55)
Outliers: 183324 (41.7%)


In [24]:
# Verify outliers from full transform still receive meaningful probabilities
outlier_indices = np.where(np.array(all_pros_topics) == -1)[0]
sample_outliers = np.random.choice(outlier_indices, size=5)
topic_info = pros_topic_model.get_topic_info()

for idx in sample_outliers:
    print(f"Review: {train.loc[idx, 'pros_clean'][:100]}...")
    print(f"Word count: {train.loc[idx, 'pros_words']}")
    
    # Get top 5 topic probabilities
    probs = all_pros_probs[idx]
    top_topics = np.argsort(probs)[::-1][:5]
    
    print("Top 5 topic probabilities:")
    for t in top_topics:
        topic_name = topic_info.loc[topic_info['Topic'] == t, 'Name'].values[0]
        print(f"  Topic {t}: {probs[t]:.3f} - {topic_name}")
    print("=" * 60)
    print()

Review: realloy enjoyed my time there ok...
Word count: 6
Top 5 topic probabilities:
  Topic 27: 0.184 - 27_bank_banking_banks_financial
  Topic 30: 0.000 - 30_travel_world_haul_benefits
  Topic 25: 0.000 - 25_ceo_vision_great_new
  Topic 38: 0.000 - 38_flexi_time_flexitime_leave
  Topic 9: 0.000 - 9_london_uk_office_good

Review: they definitely do value and care about their people and give you as much support as you need for yo...
Word count: 26
Top 5 topic probabilities:
  Topic 44: 0.118 - 44_team_im_feel_support
  Topic 19: 0.045 - 19_organisation_great_team_atmosphere
  Topic 36: 0.043 - 36_environment_friendly_staff_work
  Topic 42: 0.040 - 42_management_progression_role_opportunities
  Topic 29: 0.038 - 29_people_great_clients_lots

Review: - compensation and benefits; - freedom to choose how you solve the problems; - international team, i...
Word count: 67
Top 5 topic probabilities:
  Topic 32: 0.055 - 32_good_flexible_environment_working
  Topic 18: 0.044 - 18_balance_life_wo

Even reviews classified as outliers receive interpretable probability distributions. Short reviews (<10 words) have sparser probabilities, while longer reviews map clearly to relevant topics. This confirms that using topic probabilities as features is valid for all reviews, not just those assigned to specific clusters.

## 4.8 Fit BERTopic on Cons

Same pipeline as pros: filter 20+ words → cap per company (65) → fit → transform all.

In [25]:
# Fit BERTopic on cons

# Filter to 20+ word cons (cons_words already computed in 4.7)
cons_20plus = train[train["cons_words"] >= 20].copy()

# Cap per company for balanced representation
max_per_company = 65
cons_capped = cons_20plus.groupby('firm').apply(
    lambda x: x.sample(min(len(x), max_per_company), random_state=42)
).reset_index(drop=True)

# Extract docs and embeddings using tracked indices
docs_long_cons = cons_capped["cons_clean"].tolist()
emb_long_cons = cons_embeddings["train"][cons_capped["emb_idx"].values]

print(f"20+ word cons: {len(cons_20plus)} ({len(cons_20plus)/len(train)*100:.1f}%)")
print(f"After capping at {max_per_company}/company: {len(cons_capped)} ({len(cons_capped)/len(train)*100:.1f}%)")

# Fit BERTopic with same settings as pros
vectorizer = CountVectorizer(stop_words='english')
umap_model = UMAP(n_neighbors=5, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, min_samples=10, prediction_data=True)

cons_topic_model = BERTopic(
    vectorizer_model=vectorizer,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True
)

print("\nFitting BERTopic on cons...")
t0 = time.time()
cons_topics_fit, cons_probs_fit = cons_topic_model.fit_transform(docs_long_cons, embeddings=emb_long_cons)
elapsed = time.time() - t0

n_topics = len(set(cons_topics_fit)) - (1 if -1 in cons_topics_fit else 0)
n_outliers = sum(1 for t in cons_topics_fit if t == -1)
print(f"\nFit complete in {elapsed:.0f}s")
print(f"Cons topics discovered: {n_topics}")
print(f"Outliers: {n_outliers} ({n_outliers/len(cons_topics_fit)*100:.1f}%)")

# Show ALL cons topics
print(f"\nAll {n_topics} Cons topics:")
for _, row in cons_topic_model.get_topic_info().iterrows():
    if row["Topic"] != -1:
        print(f"  Topic {row['Topic']:3d}: {row['Count']:5d} docs - {row['Name']}")

2026-02-28 16:45:44,773 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


20+ word cons: 128847 (29.3%)
After capping at 65/company: 19001 (4.3%)

Fitting BERTopic on cons...


2026-02-28 16:45:55,243 - BERTopic - Dimensionality - Completed ✓
2026-02-28 16:45:55,245 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-28 16:46:02,491 - BERTopic - Cluster - Completed ✓
2026-02-28 16:46:02,500 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-28 16:46:03,574 - BERTopic - Representation - Completed ✓



Fit complete in 20s
Cons topics discovered: 62
Outliers: 8360 (44.0%)

All 62 Cons topics:
  Topic   0:   618 docs - 0_university_students_research_teaching
  Topic   1:   570 docs - 1_senior_management_leadership_staff
  Topic   2:   549 docs - 2_store_stores_customers_shop
  Topic   3:   501 docs - 3_bullying_staff_management_managers
  Topic   4:   457 docs - 4_lack_poor_management_staff
  Topic   5:   413 docs - 5_hr_team_people_management
  Topic   6:   402 docs - 6_shifts_shift_hours_days
  Topic   7:   387 docs - 7_promotion_promoted_promotions_years
  Topic   8:   368 docs - 8_training_job_new_learn
  Topic   9:   339 docs - 9_cons_think_really_working
  Topic  10:   324 docs - 10_career_progression_opportunities_development
  Topic  11:   292 docs - 11_sick_pay_contract_days
  Topic  12:   282 docs - 12_hours_long_work_balance
  Topic  13:   236 docs - 13_hotel_hotels_guests_guest
  Topic  14:   227 docs - 14_salary_pay_salaries_competitive
  Topic  15:   211 docs - 15_overti

In [26]:
# --- Transform all cons to get probabilities ---
print(f"Transforming all {len(train)} cons...")
t0 = time.time()
all_cons_topics, all_cons_probs = cons_topic_model.transform(
    train["cons_clean"].tolist(),
    embeddings=cons_embeddings["train"]
)
elapsed = time.time() - t0

print(f"Transform complete in {elapsed:.0f}s")
print(f"\n--- CONS SUMMARY ---")
print(f"Number of cons topics: {len(cons_topic_model.get_topic_info()) - 1}")
print(f"Probability matrix shape: {np.array(all_cons_probs).shape}")
n_outliers = sum(1 for t in all_cons_topics if t == -1)
print(f"Outliers: {n_outliers} ({n_outliers/len(all_cons_topics)*100:.1f}%)")

# --- COMBINED SUMMARY ---
print(f"\n{'=' * 60}")
print(f"COMBINED SUMMARY")
print(f"{'=' * 60}")
print(f"Pros topics: {len(pros_topic_model.get_topic_info()) - 1}")
print(f"Cons topics: {len(cons_topic_model.get_topic_info()) - 1}")
print(f"Pros probability matrix: {np.array(all_pros_probs).shape}")
print(f"Cons probability matrix: {np.array(all_cons_probs).shape}")
print(f"Total topic features: {np.array(all_pros_probs).shape[1] + np.array(all_cons_probs).shape[1]}")

2026-02-28 16:47:20,652 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.


Transforming all 440112 cons...


2026-02-28 16:49:05,614 - BERTopic - Dimensionality - Completed ✓
2026-02-28 16:49:05,615 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-02-28 16:49:32,852 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-02-28 16:52:15,496 - BERTopic - Probabilities - Completed ✓
2026-02-28 16:52:15,498 - BERTopic - Cluster - Completed ✓


Transform complete in 296s

--- CONS SUMMARY ---
Number of cons topics: 62
Probability matrix shape: (440112, 62)
Outliers: 217563 (49.4%)

COMBINED SUMMARY
Pros topics: 55
Cons topics: 62
Pros probability matrix: (440112, 55)
Cons probability matrix: (440112, 62)
Total topic features: 117


## 4.9 Transform Val and Test

Apply the fitted pros and cons topic models to val/test splits to get topic probability features.

In [27]:
# Transform val
print("Transforming val pros...")
val_pros_topics, val_pros_probs = pros_topic_model.transform(
    val["pros_clean"].tolist(), embeddings=pros_embeddings["val"]
)
print("Transforming val cons...")
val_cons_topics, val_cons_probs = cons_topic_model.transform(
    val["cons_clean"].tolist(), embeddings=cons_embeddings["val"]
)

# Transform test
print("Transforming test pros...")
test_pros_topics, test_pros_probs = pros_topic_model.transform(
    test["pros_clean"].tolist(), embeddings=pros_embeddings["test"]
)
print("Transforming test cons...")
test_cons_topics, test_cons_probs = cons_topic_model.transform(
    test["cons_clean"].tolist(), embeddings=cons_embeddings["test"]
)

print(f"\nVal pros: {np.array(val_pros_probs).shape}, cons: {np.array(val_cons_probs).shape}")
print(f"Test pros: {np.array(test_pros_probs).shape}, cons: {np.array(test_cons_probs).shape}")

2026-02-28 17:09:45,892 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.


Transforming val pros...


2026-02-28 17:10:00,635 - BERTopic - Dimensionality - Completed ✓
2026-02-28 17:10:00,636 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-02-28 17:10:04,015 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-02-28 17:10:19,737 - BERTopic - Probabilities - Completed ✓
2026-02-28 17:10:19,738 - BERTopic - Cluster - Completed ✓
2026-02-28 17:10:19,874 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.


Transforming val cons...


2026-02-28 17:10:35,208 - BERTopic - Dimensionality - Completed ✓
2026-02-28 17:10:35,209 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-02-28 17:10:38,649 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-02-28 17:10:55,430 - BERTopic - Probabilities - Completed ✓
2026-02-28 17:10:55,430 - BERTopic - Cluster - Completed ✓
2026-02-28 17:10:55,594 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.


Transforming test pros...


2026-02-28 17:11:09,457 - BERTopic - Dimensionality - Completed ✓
2026-02-28 17:11:09,458 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-02-28 17:11:12,642 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-02-28 17:11:28,020 - BERTopic - Probabilities - Completed ✓
2026-02-28 17:11:28,020 - BERTopic - Cluster - Completed ✓
2026-02-28 17:11:28,147 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.


Transforming test cons...


2026-02-28 17:11:42,138 - BERTopic - Dimensionality - Completed ✓
2026-02-28 17:11:42,140 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-02-28 17:11:45,613 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-02-28 17:12:02,497 - BERTopic - Probabilities - Completed ✓
2026-02-28 17:12:02,498 - BERTopic - Cluster - Completed ✓



Val pros: (94310, 55), cons: (94310, 62)
Test pros: (94310, 55), cons: (94310, 62)


## 4.10 Topic Feature EDA

Explore discovered topics and their association with employee recommendations.

In [33]:
# Semantic theme groupings
pros_themes = {
    "Compensation & Benefits": [0, 6, 22, 23],
    "Culture & People": [8, 11, 15, 19, 29, 36, 44, 52],
    "Work-Life Balance": [10, 18, 32, 38, 43, 45, 48],
    "Career & Growth": [7, 17, 26, 31, 34, 42, 50],
    "Leadership": [5, 25],
    "Environment & Location": [3, 9, 20, 40, 53],
    "Company & Brand": [4, 24, 28],
    "Industry-Specific": [1, 2, 12, 27, 46, 51, 54],
    "Generic/Unclear": [13, 14, 16, 21, 30, 33, 35, 37, 39, 41, 47, 49],
}

cons_themes = {
    "Poor Management": [1, 4, 5, 17, 28, 33, 44, 55],
    "Toxic Culture": [3, 24, 39, 41, 59],
    "Compensation Issues": [11, 14, 15, 26, 34, 48, 50, 51],
    "Work-Life Balance": [6, 12, 22, 38, 46, 49, 61],
    "Career Limitations": [7, 8, 10],
    "Systems & Bureaucracy": [18, 40, 52, 60],
    "Environment & Location": [21, 27, 30, 35],
    "Industry-Specific": [0, 2, 13, 20, 23, 25, 43, 47, 54, 58],
    "Generic/Unclear": [9, 16, 19, 29, 31, 32, 36, 37, 42, 45, 53, 56, 57],
}

# Aggregate topic probabilities by theme
pros_probs_arr = np.array(all_pros_probs)
cons_probs_arr = np.array(all_cons_probs)
recommend_mask = train["recommend_binary"] == 1

# Calculate theme probabilities (sum of topic probs in each theme)
def get_theme_probs(probs_arr, themes):
    theme_probs = {}
    for theme, topic_ids in themes.items():
        theme_probs[theme] = probs_arr[:, topic_ids].sum(axis=1)
    return theme_probs

pros_theme_probs = get_theme_probs(pros_probs_arr, pros_themes)
cons_theme_probs = get_theme_probs(cons_probs_arr, cons_themes)

# Compare mean theme probability by recommendation
print("=" * 70)
print("PROS THEMES: Mean probability by recommendation")
print("=" * 70)
print(f"{'Theme':<25} {'Recommend':>10} {'Not Rec':>10} {'Diff':>10}")
print("-" * 70)

pros_results = []
for theme, probs in pros_theme_probs.items():
    mean_rec = probs[recommend_mask].mean()
    mean_not = probs[~recommend_mask].mean()
    diff = mean_rec - mean_not
    pros_results.append((theme, mean_rec, mean_not, diff))

pros_results.sort(key=lambda x: x[3], reverse=True)
for theme, mean_rec, mean_not, diff in pros_results:
    print(f"{theme:<25} {mean_rec:>10.3f} {mean_not:>10.3f} {diff:>+10.3f}")

print("\n" + "=" * 70)
print("CONS THEMES: Mean probability by recommendation")
print("=" * 70)
print(f"{'Theme':<25} {'Recommend':>10} {'Not Rec':>10} {'Diff':>10}")
print("-" * 70)

cons_results = []
for theme, probs in cons_theme_probs.items():
    mean_rec = probs[recommend_mask].mean()
    mean_not = probs[~recommend_mask].mean()
    diff = mean_rec - mean_not
    cons_results.append((theme, mean_rec, mean_not, diff))

cons_results.sort(key=lambda x: x[3], reverse=True)
for theme, mean_rec, mean_not, diff in cons_results:
    print(f"{theme:<25} {mean_rec:>10.3f} {mean_not:>10.3f} {diff:>+10.3f}")

PROS THEMES: Mean probability by recommendation
Theme                      Recommend    Not Rec       Diff
----------------------------------------------------------------------
Company & Brand                0.068      0.056     +0.013
Career & Growth                0.116      0.105     +0.011
Culture & People               0.122      0.112     +0.010
Work-Life Balance              0.129      0.122     +0.007
Environment & Location         0.054      0.050     +0.003
Leadership                     0.013      0.011     +0.002
Industry-Specific              0.051      0.064     -0.013
Generic/Unclear                0.082      0.103     -0.021
Compensation & Benefits        0.089      0.113     -0.024

CONS THEMES: Mean probability by recommendation
Theme                      Recommend    Not Rec       Diff
----------------------------------------------------------------------
Generic/Unclear                0.185      0.138     +0.047
Work-Life Balance              0.110      0.088     +

**Observations:**

Theme-level differences are small (all < 0.05), so these are directional patterns rather than strong effects. Tentatively:

- *Pros:* Recommenders slightly more likely to discuss culture, career growth, and company brand; non-recommenders slightly more focused on compensation
- *Cons:* Non-recommenders more likely to mention poor management and toxic culture; "Generic/Unclear" for recommenders likely reflects "can't think of cons" reviews

These patterns are suggestive but modest — predictive power will come from combining all topic features with sub-ratings in the classification models.

## 4.11 Save Topic Features

Concatenate pros + cons topic probabilities into a single feature matrix per split. Save to disk along with the fitted BERTopic models.

In [34]:
# Concatenate pros + cons topic probabilities for each split
train_topic_features = np.concatenate([np.array(all_pros_probs), np.array(all_cons_probs)], axis=1)
val_topic_features = np.concatenate([np.array(val_pros_probs), np.array(val_cons_probs)], axis=1)
test_topic_features = np.concatenate([np.array(test_pros_probs), np.array(test_cons_probs)], axis=1)

print(f"Topic feature shapes:")
print(f"  Train: {train_topic_features.shape}")
print(f"  Val:   {val_topic_features.shape}")
print(f"  Test:  {test_topic_features.shape}")

# Get topic labels from models
pros_info = pros_topic_model.get_topic_info()
cons_info = cons_topic_model.get_topic_info()
pros_labels = pros_info[pros_info['Topic'] != -1]['Name'].tolist()
cons_labels = cons_info[cons_info['Topic'] != -1]['Name'].tolist()

# Create feature names
topic_feature_names = [f"pros_{label}" for label in pros_labels] + \
                      [f"cons_{label}" for label in cons_labels]

# Save topic features
np.save("../data/processed/train_topic_features.npy", train_topic_features)
np.save("../data/processed/val_topic_features.npy", val_topic_features)
np.save("../data/processed/test_topic_features.npy", test_topic_features)
np.save("../data/processed/topic_feature_names.npy", topic_feature_names)

# Save fitted BERTopic models
os.makedirs("../models", exist_ok=True)
pros_topic_model.save("../models/bertopic_pros")
cons_topic_model.save("../models/bertopic_cons")

print(f"\nSaved:")
print(f"  - train/val/test_topic_features.npy")
print(f"  - topic_feature_names.npy ({len(topic_feature_names)} features)")
print(f"  - models/bertopic_pros and models/bertopic_cons")

Topic feature shapes:
  Train: (440112, 117)
  Val:   (94310, 117)
  Test:  (94310, 117)


2026-02-28 17:34:43,147 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
2026-02-28 17:34:43,518 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.



Saved:
  - train/val/test_topic_features.npy
  - topic_feature_names.npy (117 features)
  - models/bertopic_pros and models/bertopic_cons


In [35]:
# Drop temporary columns, keep pros/cons for error analysis later
cols_to_drop = ['firm', 'pros_clean', 'cons_clean', 
                'emb_idx', 'pros_words', 'cons_words']

train = train.drop(columns=[c for c in cols_to_drop if c in train.columns])
val = val.drop(columns=[c for c in cols_to_drop if c in val.columns])
test = test.drop(columns=[c for c in cols_to_drop if c in test.columns])

# Re-save CSVs
train.to_csv("../data/processed/train.csv", index=False)
val.to_csv("../data/processed/val.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

# Verify
print("Saved CSVs:")
print("=" * 50)
for name, path in [('Train', '../data/processed/train.csv'), 
                   ('Val', '../data/processed/val.csv'), 
                   ('Test', '../data/processed/test.csv')]:
    df = pd.read_csv(path, nrows=1)
    print(f"{name}: {len(df.columns)} columns")

print(f"\nColumns: {df.columns.tolist()}")

Saved CSVs:
Train: 27 columns
Val: 27 columns
Test: 27 columns

Columns: ['overall_rating', 'work_life_balance', 'culture_values', 'career_opp', 'comp_benefits', 'senior_mgmt', 'pros', 'cons', 'recommend_binary', 'firm_encoded', 'is_current', 'ceo_approv_Disapprove', 'ceo_approv_Mild', 'ceo_approv_No Opinion', 'seniority_level_Management', 'seniority_level_Other', 'seniority_level_Senior', 'seniority_level_Unknown', 'review_year_2013', 'review_year_2014', 'review_year_2015', 'review_year_2016', 'review_year_2017', 'review_year_2018', 'review_year_2019', 'review_year_2020', 'review_year_2021']


## 4.12 Final Summary

In [36]:
print("=" * 60)
print("TEXT FEATURE ENGINEERING COMPLETE")
print("=" * 60)

n_pros = len(pros_topic_model.get_topic_info()) - 1
n_cons = len(cons_topic_model.get_topic_info()) - 1

print(f"\nTopics discovered:")
print(f"  Pros: {n_pros}")
print(f"  Cons: {n_cons}")
print(f"  Total topic features: {n_pros + n_cons}")

print(f"\nTopic feature shapes:")
print(f"  Train: {train_topic_features.shape}")
print(f"  Val:   {val_topic_features.shape}")
print(f"  Test:  {test_topic_features.shape}")

print(f"\nCSV columns: {len(train.columns)}")

print(f"\nFiles saved:")
print(f"  data/processed/train.csv, val.csv, test.csv")
print(f"  data/processed/train_topic_features.npy, val_topic_features.npy, test_topic_features.npy")
print(f"  data/processed/topic_feature_names.npy")
print(f"  models/bertopic_pros/, models/bertopic_cons/")

print(f"\nReady for Stage 5: Model Exploration!")

TEXT FEATURE ENGINEERING COMPLETE

Topics discovered:
  Pros: 55
  Cons: 62
  Total topic features: 117

Topic feature shapes:
  Train: (440112, 117)
  Val:   (94310, 117)
  Test:  (94310, 117)

CSV columns: 27

Files saved:
  data/processed/train.csv, val.csv, test.csv
  data/processed/train_topic_features.npy, val_topic_features.npy, test_topic_features.npy
  data/processed/topic_feature_names.npy
  models/bertopic_pros/, models/bertopic_cons/

Ready for Stage 5: Model Exploration!
